# 08 — MLflow Tracking (Phase 6)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent) if Path.cwd().name=='notebooks' else str(Path.cwd()))
from src.ml.tracking import init_tracking, log_run
import pandas as pd

init_tracking('industrial-iot-pm')
print('tracking uri:', __import__('mlflow').get_tracking_uri())


In [ ]:
# Log refined LOTO results
for farm in ['A','B','C']:
    df = pd.read_csv(f'reports/loto_refined_{farm}.csv')
    for _, row in df.iterrows():
        params = {'farm': farm, 'model': row['model'], 'contamination': row.get('contamination'),
                  'calibrated': row.get('calibrated'), 'threshold': row.get('threshold'), 'n_features': 10}
        metrics = {'precision': row['precision'], 'recall': row['recall'], 'f1': row['f1'],
                   'false_alarms_per_day': row['false_alarms_per_day'],
                   'event_detection_rate': row['event_detection_rate']}
        log_run(f'loto_{farm}_{row["model"]}_{row.get("contamination")}', params, metrics)
print('logged')


In [ ]:
# Query experiments
import mlflow
exp = mlflow.get_experiment_by_name('industrial-iot-pm')
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id])
print('runs:', len(runs))
print(runs[['tags.mlflow.runName','metrics.f1','metrics.recall']].head(10).to_string(index=False))
